# Calvin teaching follow-up

Run the cells in order after the synthetic A/B lesson. Predict each result before running its cell. The three exercises use invented inputs and work without the Site 1 study bundle.

In [ ]:
import json
import math
import os
from copy import deepcopy
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from deeprootgen.run_records import tracked_run
from deeprootgen.statistics.spatial_rld import calculate_core_rld

cwd = Path.cwd()
if "DRG_PROJECT_ROOT" in os.environ:
    root = Path(os.environ["DRG_PROJECT_ROOT"])
else:
    root = next(p for p in (cwd, *cwd.parents) if (p / "deeprootgen").is_dir())
fixture = root / "examples/design/native_crown_0_20cm.json"
output = Path(os.environ.get("DRG_LESSON_OUTPUT", str(root / ".runtime")))
output.mkdir(parents=True, exist_ok=True)
lesson = Path(mkdtemp(prefix="calvin-followup-", dir=output))
print(lesson)

## D. Why was a daughter accepted, capped or rejected?

Set the same proposed order-2 starting diameter in all three runs. Propose one site 1 cm along the primary. First switch on the 0.8 daughter cap. Then change only the order-1 taper rate, keeping the cap on. Predict which case has a smaller daughter and which has none.

The high taper rate is intentionally exaggerated for this short root. The table shows the expected point diameters from the taper and the actual branch counts from the generator. Open each case's `parameters.json`, `root_summary.json` and `organs.csv` to trace the result.

In [ ]:
base = json.loads(fixture.read_text())
base["parameters"].update(primary_root_count=1, outer_primary_fraction=1.0,
                          min_branching_diameter_cm=0.093)
features = base["features"]
linear = dict(law="piecewise_linear_arclength", base_cm=0.18, tip_cm=0.02)
primary = dict(law="exponential_above_diameter", base_cm=0.20, tip_cm=0.02,
               rate_per_cm=0.05, thick_min_cm=0.07)
features["taper"] = dict(enabled=True, settings={"by_order": {
    "1": primary, "2": linear, "3": deepcopy(linear),
}})
features["branching"] = dict(enabled=True, settings={"by_order": {
    "1": dict(law="first_branch_and_daughter_count", branch_probability=1.0,
              first_branch_cm=dict(mean=1.0, sd=0.0, min=1.0, max=1.0),
              daughter_probabilities=[1.0]),
}})
base["parameters"].pop("daughters_per_parent", None)
base["parameters"]["daughters_by_parent_order"] = {"2": 0}

rows = []
for label, rate, cap_on in (
    ("D1_no_cap", 0.05, False),
    ("D2_cap", 0.05, True),
    ("D3_faster_taper", 1.0, True),
):
    config = deepcopy(base)
    config["features"]["taper"]["settings"]["by_order"]["1"]["rate_per_cm"] = rate
    config["features"]["parent_diameter_cap"] = dict(
        enabled=cap_on, settings=dict(fraction_max=0.8)
    )
    destination = lesson / label
    tracked_run(config, destination, label=label, ledger=lesson / "runs.jsonl")
    summary = json.loads((destination / "root_summary.json").read_text())
    daughters = next(row["root_count"] for row in summary["by_order"] if row["order"] == 2)
    organs = pd.read_csv(destination / "organs.csv")
    arc = config["features"]["branching"]["settings"]["by_order"]["1"]["first_branch_cm"]["mean"]
    parent_mm = 10 * primary["base_cm"] * math.exp(-rate * arc)
    daughter_mm = min(1.8, 0.8 * parent_mm) if cap_on else 1.8
    rows.append({
        "case": label,
        "site set by input (cm)": arc,
        "accepted daughters": daughters,
        "daughter mass (g)": organs.loc[organs["order"] == 2, "mass_g"].sum(),
        "expected parent at site (mm)": parent_mm,
        "expected daughter base (mm)": daughter_mm if daughters else None,
    })

diameter_cases = pd.DataFrame(rows)
display(diameter_cases.round(3))

## E. Where does the virtual core intersect the root?

An invented 10 cm root segment runs from x = -5 to +5 at y = 0 and z = -5 cm. Use the same segment for three cylinders. Move a narrow core 2 cm away from the segment, then widen the core at that offset. Predict the clipped root length and what happens to RLD when the denominator grows.

This exercise uses the package's classroom cylinder calculation with no lower diameter filter. Site 1 uses its own paired-core observation operator and diameter band. A synthetic miss here is a true zero intersection. A missing field sample is a different case.

In [ ]:
nodes = pd.DataFrame([
    dict(node_id=0, parent_id=-1, order=0, x=-5.0, y=0.0, z=-5.0,
         diameter=0.0, length=0.0),
    dict(node_id=1, parent_id=0, order=1, x=5.0, y=0.0, z=-5.0,
         diameter=0.15, length=10.0),
])
cores = [("centred", 0.0, 1.5), ("offset", 2.0, 1.5), ("offset, wider", 2.0, 3.0)]
rows = []
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True, sharey=True)
for ax, (name, y, radius) in zip(axes, cores):
    rld = calculate_core_rld(nodes, x=0.0, y=y, z_lower_bound=-10.0,
                             z_upper_bound=0.0, core_radius=radius)
    volume = math.pi * radius**2 * 10
    rows.append(dict(core=name, radius_cm=radius, soil_volume_cm3=volume,
                     clipped_length_cm=rld * volume, rld_cm_per_cm3=rld))
    ax.plot([-5, 5], [0, 0], color="#986849", linewidth=3)
    ax.add_patch(plt.Circle((0, y), radius, color="#277e85", alpha=0.28))
    ax.set(title=name, xlim=(-5.5, 5.5), ylim=(-3.5, 5.5), xlabel="x (cm)")
    ax.set_aspect("equal")
axes[0].set_ylabel("y (cm)")
fig.tight_layout()
plt.show()
core_cases = pd.DataFrame(rows)
display(core_cases.round(4))

## F. Which candidate would you advance?

These invented rows mimic the Site 1 review. Compare candidates on the shared seed panel. Find the lowest RLD score and then check mean predicted/observed RLD and the fraction of vines between 5 and 7 kg. Which candidate would you advance to fresh seeds?

For this exercise, the mean RLD ratio range is 0.8–1.2, the mean mass range is 5–7 kg, and the provisional fresh-seed mass check is 90% per realization. A four-seed panel is too small to establish that last rate reliably. Use it here to understand the denominator. The Site 1 notebook has the actual developed candidates, comparison support and uncertainty when available.

In [ ]:
trials = pd.DataFrame({
    "candidate": ["lower RLD error"] * 4 + ["balanced"] * 4,
    "seed": [101, 102, 103, 104] * 2,
    "illustrative_rld_score": [0.38, 0.47, 0.45, 0.46, 0.51, 0.49, 0.50, 0.52],
    "rld_ratio": [0.95, 0.91, 1.02, 0.98, 0.86, 0.92, 0.95, 1.03],
    "total_dry_mass_kg": [4.3, 5.2, 5.5, 7.3, 5.3, 5.8, 6.1, 6.5],
})
trials["mass_in_band"] = trials["total_dry_mass_kg"].between(5, 7)
comparison = trials.groupby("candidate", sort=False).agg(
    mean_example_score=("illustrative_rld_score", "mean"),
    mean_rld_ratio=("rld_ratio", "mean"),
    mean_mass_kg=("total_dry_mass_kg", "mean"),
    fraction_in_band=("mass_in_band", "mean"),
    seeds=("seed", "size"),
)
display(trials)
display(comparison.round(3))

## What to show in the existing interface

- **Simulation:** load D1 and D2's saved `parameters.json` files and find the single cap change in **Advanced settings (JSON)**. Use the same seed. Show the saved root summaries and organ tables beside the form. If the local worker has been rehearsed, rerun a synthetic case and export it.
- **EDA:** load the synthetic `nodes.csv` files in turn. Plot x against z and group by order to see the missing daughter in D3. Use the 0.93 mm gate and calculated point diameter to explain the missing order. The diameter column describes the exported segment's equivalent diameter, so use the taper formula to teach the point diameter at the attachment.
- **Next interface addition:** after a completed simulation, show the active taper and cap, daughter proposals, diameter rejections, accepted daughters and mass by diameter class. Read the root summary plus the branch receipt on builds that export it. A later two-run comparison could show which input changed and compare the outputs at a common seed.
- **Virtual core and candidate review:** keep E and F in Jupyter for the lesson. A future core preview could overlay one cylinder and show clipped length, soil volume and RLD. For field ranking, use the existing Site 1 notebook and its matched cores.